###Silver Layer (Initial Cleaning & Staging Data)
1. Initial Cleansing encompass 
  - investigate total rows (55,000 entries), duplicates (534 entries), null values (empty)
  - Revamp data type (Billing Amount from double into decimal ) and format name(initcap)
  - Add up new columns (Estimated_Length_of_stay & Patient_id)
  - Do Regular Expression (Regexp) : get rid of many noises (_)(-)(,)(and)(Ltd,Plc,Inc,Group)
  - Remove column name for security & storage
2. Staging Data
  - Staging Data Table


In [0]:
df_silver = spark.read.table("Bronze_Data_Table")
display(df_silver.limit(10))

In [0]:
total_rows = df_silver.count()
print(f"Silver layer total rows: {total_rows}")

In [0]:
from pyspark.sql.functions import col
silver_duplicate = df_silver.groupBy(df_silver.columns).count().filter(col("count") > 1).count()

display(silver_duplicate)

In [0]:
df_silver_clean = df_silver.dropDuplicates()
print(f"Silver layer total rows after removing duplicates: {df_silver_clean.count()}")

In [0]:
from pyspark.sql.functions import expr

df_silver1 = df_silver_clean.withColumn("Name", expr("initcap(Name)"))

display(df_silver1)

In [0]:
df_silver1.isEmpty()

In [0]:
from pyspark.sql.functions import col, expr, datediff, trim, initcap, regexp_replace

df_silver_id = df_silver1.withColumn(
    "Billing_Amount", expr("cast(`Billing_Amount` as decimal(10,2))")).withColumn(
    "Estimated_Length_of_Stay", datediff(col("Discharge_Date"), col("Date_of_Admission"))).withColumn(
    "Patient_ID", expr("uuid()"))
    
df_silver_id.show(10,False)

In [0]:
from pyspark.sql.functions import col, regexp_replace, split

#for noise - and (and) at front of hospital name (1)
# for Fist name of hospitals can be used (2)
# Delete spaces 
df_silver_regex = df_silver_id. \
withColumn("Hospital", regexp_replace(col("Hospital"), r"^(?!)(?:-s\*|and\s+)+","")) \
.withColumn("Hospital", regexp_replace(col("Hospital"), r"(?i)\b(and|Ltd|Llc|Inc|Plc|Group)\b", "")).withColumn("Hospital", regexp_replace(col("Hospital"), r",", "")).withColumn("Hospital", regexp_replace(col("Hospital"), r"-", " ")) \
.withColumn("Hospital", trim(regexp_replace(col("Hospital"), r"\s+", " "))).withColumn("Hospital", split(col("Hospital"), " ")[0]) 

display(df_silver_regex)


In [0]:
from pyspark.sql.functions import concat, lit

df_silver_regex = df_silver_regex.withColumn("Hospital", trim(regexp_replace(col("Hospital"), r"(?i)\bHospital\b", ""))
) \
.withColumn("Hospital", concat(col("Hospital"), lit(" Hospital"))) \
.withColumn("Insurance_Provider", regexp_replace(col("Insurance_Provider"), "UnitedHealthcare", "United Healthcare")) \
.withColumn("Doctor", regexp_replace(col("Doctor"), r"(?i)\b(Mr\.|Mrs\.|Ms\.|Dr\.|Miss)\s+", "")
).drop("Name")

display(df_silver_regex)

In [0]:
df_silver_regex.write.mode("overwrite").format("delta").saveAsTable("Staging_Table")